# Prevent the Overdraft — Transactions Hands-On

**DBPRA · TU Berlin**

Two payments race against the same account balance. The naive code below lets the balance go *below zero* even though the application is supposed to enforce **no overdraft (`balance ≥ 0`)**. Your job: fix it with the right isolation level + retry loop.

> Run the cells from top to bottom. The first cell takes ~30s the first time (it installs PostgreSQL on the Colab VM).

## Setup

Install PostgreSQL, start the service, create a `root` superuser so psycopg2 can connect over the Unix socket via peer auth.

In [ ]:
!sudo apt-get -qq install -y postgresql > /dev/null
!sudo service postgresql start
!sudo -u postgres psql -c "CREATE USER root SUPERUSER;" 2>/dev/null
!sudo -u postgres createdb demo 2>/dev/null
!pip install -q psycopg2-binary

In [ ]:
import psycopg2
from psycopg2 import errors as pgerr
import threading
import time

DSN = "dbname=demo"

def reset_db():
    con = psycopg2.connect(DSN); con.autocommit = True
    cur = con.cursor()
    cur.execute("DROP TABLE IF EXISTS customer")
    cur.execute("CREATE TABLE customer (c_custkey INT PRIMARY KEY, c_acctbal NUMERIC)")
    cur.execute("INSERT INTO customer VALUES (17, 150)")
    con.close()

def balance(account_id=17):
    con = psycopg2.connect(DSN)
    cur = con.cursor()
    cur.execute("SELECT c_acctbal FROM customer WHERE c_custkey = %s", (account_id,))
    bal = cur.fetchone()[0]
    con.close()
    return bal

def run_concurrent(fn, n=2):
    results = []
    def worker():
        results.append(fn())
    threads = [threading.Thread(target=worker) for _ in range(n)]
    for t in threads: t.start()
    for t in threads: t.join()
    return results

reset_db()
print("Starting balance:", balance())

## The naive withdrawal

The function below reads the balance, checks it's enough, then writes the new value. No transaction control — each statement runs in **autocommit** mode.

The `time.sleep(0.05)` only widens the race window so the bug is reliably visible in this demo. In a real workload the race is rarer but very real.

In [ ]:
def withdraw_naive(amount, account_id=17):
    con = psycopg2.connect(DSN); con.autocommit = True
    cur = con.cursor()
    cur.execute("SELECT c_acctbal FROM customer WHERE c_custkey = %s", (account_id,))
    bal = cur.fetchone()[0]

    time.sleep(0.05)  # widen race window

    if bal < amount:
        con.close()
        return "rejected"

    cur.execute("UPDATE customer SET c_acctbal = c_acctbal - %s WHERE c_custkey = %s",
                (amount, account_id))
    con.close()
    return "ok"

reset_db()
results = run_concurrent(lambda: withdraw_naive(100), n=2)
print("Results:        ", results)
print("Final balance:  ", balance())
# Expect: both 'ok', balance = -50  →  OVERDRAFT

### Your task

Implement `withdraw_safe(amount, account_id=17)` so the invariant `balance ≥ 0` holds even under concurrent withdrawals.

You need to:

- Set the isolation level to **`SERIALIZABLE`** at the start of the transaction
- **Retry** when PostgreSQL raises a `SerializationFailure` (it will: that's how it tells you "your transaction would have broken serializability")
- Return `"ok"` (succeeded), `"rejected"` (insufficient funds), or give up after a few retries

Hints:

```python
con.set_session(isolation_level="SERIALIZABLE")
...
except pgerr.SerializationFailure:
    # rollback, sleep, try again
```

In [ ]:
def withdraw_safe(amount, account_id=17, max_retries=5):
    for attempt in range(max_retries):
        try:
            con = psycopg2.connect(DSN)
            con.set_session(isolation_level="SERIALIZABLE")
            cur = con.cursor()

            # TODO: SELECT the balance
            # TODO: keep the time.sleep(0.05) for a fair comparison
            # TODO: if balance < amount, rollback + return "rejected"
            # TODO: UPDATE the balance, commit, return "ok"

            con.close()
            return "TODO"
        except pgerr.SerializationFailure:
            con.rollback()
            con.close()
            time.sleep(0.05 * (2 ** attempt))
    return "retries exhausted"

In [ ]:
reset_db()
results = run_concurrent(lambda: withdraw_safe(100), n=2)
print("Results:        ", results)
print("Final balance:  ", balance())
# Expect: one 'ok' + one 'rejected', balance = 50  →  invariant holds

### What you proved

- `SERIALIZABLE` doesn't run transactions one at a time. It runs them concurrently, then **aborts** the ones that would have broken serializability.
- The retry loop turns those aborts into something your application can handle gracefully.
- The invariant `balance ≥ 0` is now enforced by **the database** — not just by your application's check.

This is the production pattern: explicit isolation level + retry loop, catching the precise `SerializationFailure` exception.

### Going further

- Try `n=5` concurrent withdrawals. How many `"ok"` results do you see? How many retries?
- Try replacing `SERIALIZABLE` with `READ COMMITTED` (PostgreSQL's default). Does the invariant still hold?
- Why doesn't `REPEATABLE READ` (snapshot isolation) prevent this? *(Hint: think about what the writers actually conflict on.)*